<a href="https://colab.research.google.com/github/RatchanonPa/Data-Warehouse-and-Big-Data-Analytics/blob/main/LLM_Normalize.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Download Audio File

In [3]:
!pip install pydub
!sudo apt-get install ffmpeg

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.


In [4]:
from pydub import AudioSegment

# Load the .m4a or .mp4 file
audio = AudioSegment.from_file("/content/อู้คำเมือง noised.m4a", format="m4a")

# Export as .wav
audio.export("/content/อู้คำเมือง noised.wav", format="wav")

<_io.BufferedRandom name='/content/อู้คำเมือง noised.wav'>

# Audio cleaning

In [5]:
import os
import pandas as pd
import numpy as np
from glob import glob
import audioread
import librosa
import soundfile as sf
from IPython.display import Audio
from tqdm.auto import tqdm
import json
from scipy.signal import butter, lfilter
from torch.utils.data import Dataset, DataLoader
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

In [6]:
from scipy.signal import butter, lfilter, resample_poly, find_peaks, sosfiltfilt, tf2sos, iirnotch
from fractions import Fraction
import numpy as np
from numpy import fft

def resample(y, sr):
  frac = Fraction(16000, sr).limit_denominator()
  up, down = frac.numerator, frac.denominator
  return resample_poly(y, up, down)


def compute_features(y: np.ndarray, sr: int) -> dict:
  zcr = np.mean(librosa.feature.zero_crossing_rate(y=y))
  mfcc_std = np.std(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13))
  energy = np.mean(librosa.feature.rms(y=y))
  centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
  bandwidth = np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))
  rolloff = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr, roll_percent=0.85))

  badness = (1 / (mfcc_std + 1e-5)) + (1 / (energy + 1e-5)) + zcr

  checks = [
    zcr > 0.1,
    centroid > 2500,
    bandwidth > 2000,
    rolloff > 4000,
    mfcc_std > 30,
    badness > 50,
  ]

  return sum(checks)


def remove_fft_spikes(y, sr,
                      height=800,
                      distance=50,
                      min_f=300,
                      max_f=4000,
                      Q=10.0,
                      max_notches=10):
  N = len(y)
  fft_vals = fft.fft(y)
  freqs = fft.fftfreq(N, 1/sr)
  mags = np.abs(fft_vals)

  # positive half
  half = N // 2
  freqs = freqs[:half]
  mags = mags[:half]

  peaks, props = find_peaks(mags, height=height, distance=distance)
  peak_freqs = freqs[peaks]
  peak_mags  = props["peak_heights"]

  # keep within [min_f, max_f]
  mask = (peak_freqs >= min_f) & (peak_freqs <= max_f)
  peak_freqs = peak_freqs[mask]
  peak_mags  = peak_mags[mask]

  if len(peak_freqs) == 0:
    return y

  # pick top-N by magnitude
  idx_sorted = np.argsort(peak_mags)[::-1]
  sel = idx_sorted[:max_notches]
  spike_freqs = peak_freqs[sel]

  for f0 in spike_freqs:
    y = apply_notch_filter(y, sr, f0, Q=Q)

  return y

def normalize_by_fft(y, sr, target_amp=1000, min_thresh=500, max_thresh=1300):
  mag = np.abs(np.fft.fft(y))
  peak = mag.max()
  if peak < min_thresh or peak > max_thresh:
    y = y * (target_amp / peak)
  return y

def enforce_min_fft_peak(y, sr, min_peak=5000):
    mag = np.abs(np.fft.fft(y))
    peak = mag.max()
    if peak < min_peak and peak > 0:
        y = y * (min_peak / peak)
    return y

def butter_bandpass(y, sr, lowcut=300, highcut=3400, order=4):
  nyq = sr / 2
  b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
  return lfilter(b, a, y)

def apply_notch_filter(y, sr, f0, Q=10.0):
    """
    Create a 2nd-order notch at f0, convert to SOS, and apply it twice
    for deeper attenuation.
    """
    # normalized freq
    w0 = f0 / (sr/2)
    b, a = iirnotch(w0, Q)
    sos = tf2sos(b, a)
    # apply forward-backward twice
    y = sosfiltfilt(sos, y)
    y = sosfiltfilt(sos, y)
    return y


def clean_noisy(y, sr):
  # bandpass
  y = butter_bandpass(y, sr)
  y = remove_fft_spikes(y, sr)
  y = normalize_by_fft(y, sr)
  y = enforce_min_fft_peak(y, sr)

  return y, sr


def preprocess(path):
  y, sr = sf.read(path)
  y = resample(y, sr)
  sr = 16000
  votes = compute_features(y, sr)

  if votes >= 3: # mor lam sing
    return y, sr

  if votes >= 2: # noisy, use fft to clean
    return clean_noisy(y, sr)

  return y, sr

# ASR

In [7]:
import torch
from transformers import pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

lang = "th"
task = "transcribe"

pipe = pipeline(
    task="automatic-speech-recognition",
    model="nectec/Pathumma-whisper-th-large-v3",
    torch_dtype=torch_dtype,
    device=device,
)
pipe.model.config.forced_decoder_ids = pipe.tokenizer.get_decoder_prompt_ids(language=lang, task=task)
# input_features = processor(audio_path, return_tensors="pt").input_features



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.30k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/3.93k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

...

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

Device set to use cuda


In [8]:
def speech_to_text(files_path):
  id = []
  asr_text = []
  for path in tqdm(files_path):
    idx = path.split('/')[-1].split('.')[0]
    cleaned_audio_data, sample_rate = preprocess(path)
    text = pipe(cleaned_audio_data,return_timestamps=True)["text"]
    id.append(idx)
    asr_text.append(text)
  df_asr = pd.DataFrame({'id': id, 'text': asr_text})
  return df_asr

# Inference ASR

In [9]:
df_asr_test = speech_to_text(['/content/อู้คำเมือง noised.wav'])
print(df_asr_test)

  0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/transformers/models/whisper/generation_whisper.py:573: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


                  id                                               text
0  อู้คำเมือง noised  สวัสดีเจ้า คุณ นาฏฐา ริกา เปิ้น จื้อ เต้ชินี ห...


In [10]:
from IPython.display import Audio

# กำหนด path ไปยังไฟล์ .wav ที่ผ่านการแปลงแล้ว
path = "/content/อู้คำเมือง noised.wav"

# เรียกใช้ฟังก์ชัน preprocess เพื่อ clean เสียง
cleaned_audio_data, sample_rate = preprocess(path)

# เล่นเสียงใน notebook
Audio(cleaned_audio_data, rate=sample_rate)


In [11]:
text_only = pipe(cleaned_audio_data)["text"]
print(text_only)

/usr/local/lib/python3.11/dist-packages/transformers/models/whisper/generation_whisper.py:573: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(


สวัสดีเจ้า คุณ นาฏฐา ริกา เปิ้น จื้อ เต้ชินี หรือ ได้ ซึ่ง ทราบ ว่า เต้ย ก็ได้ นะเจ้า เปน ตี้ ปรึกษา จาก ธนาคาร ไทย พาณิชย์ เจ้า วัน นี้ เต้ย จะ เข้า ม็อบ เพื่อ ฮึกฮู้ เกี่ยว กับ การ ลอง แผน ทาง การเงิน ตี้ มอง สม กับ เป้าหมาย ของ คุณ นาฏฐา ริกา เจ้า ก็ จะ มี เวลา หื้อ เต้ย สัก ก๋าว ได้ ก่ เจ้า ซัก สาม นาที ก็ได้ เจ้า


## scb10x/llama3.2-typhoon2-3b-instruct

In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# 1. Load the tokenizer and model
model_id = "scb10x/llama3.2-typhoon2-3b-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16, # Using bfloat16 for efficiency, requires compatible hardware
    device_map="auto", # Automatically maps model to available device (GPU if available, else CPU)
)

tokenizer_config.json:   0%|          | 0.00/53.1k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/778 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/303 [00:00<?, ?B/s]

In [13]:
# 2. Prepare the input messages for normalization
#    This includes a system message to set the AI's persona and task,
#    and a user message with the local dialect text to be normalized.

# Example Isan (Northeastern Thai) phrase
local_dialect_text = "สวัสดีเจ้า คุณ นาฏฐา ริกา เปิ้น จื้อ เต้ชินี หรือ ได้ ซึ่ง ทราบ ว่า เต้ย ก็ได้ นะเจ้า เปน ตี้ ปรึกษา จาก ธนาคาร ไทย พาณิชย์ เจ้า วัน นี้ เต้ย จะ เข้า ม็อบ เพื่อ ฮึกฮู้ เกี่ยว กับ การ ลอง แผน ทาง การเงิน ตี้ มอง สม กับ เป้าหมาย ของ คุณ นาฏฐา ริกา เจ้า ก็ จะ มี เวลา หื้อ เต้ย สัก ก๋าว ได้ ก่ เจ้า ซัก สาม นาที ก็ได้ เจ้า" # "Where have you been?" / "Where are you coming from?"

# You can adapt the system prompt from the model card or create a more specific one.
# For normalization, a clear instruction is key.
system_prompt_content = (
    "You are an AI assistant specialized in Thai languages. "
    "Your task is to translate local Thai dialects into Central Thai (Standard Thai). "
    "Respond directly with only the Central Thai translation of the user's message, "
    "without any additional affirmations or explanations."
)

user_prompt_content = f"Translate the following local dialect text to Central Thai: \"{local_dialect_text}\""

messages = [
    {"role": "system", "content": system_prompt_content},
    {"role": "user", "content": user_prompt_content},
]

# Apply the chat template and move to the model's device
input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True, # Important for instruction-following models
    return_tensors="pt"
).to(model.device)

# Define terminators to stop generation appropriately
terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

In [14]:
# 3. Generate the normalized text
#    Adjust generation parameters as needed.
#    For a direct translation, a lower temperature might be better.
outputs = model.generate(
    input_ids,
    max_new_tokens=100,  # Adjust based on expected length of normalized text
    eos_token_id=terminators,
    do_sample=True,      # Set to False for deterministic output if preferred
    temperature=0.3,     # Lower temperature for more focused output
    top_p=0.9,
    pad_token_id=tokenizer.eos_token_id # Suppress warning for padding
)

# Decode the response
# The response includes the input, so we slice it to get only the generated part.
response_ids = outputs[input_ids.shape[-1]:]
normalized_text = tokenizer.decode(response_ids, skip_special_tokens=True)

print(f"Local Dialect Text: {local_dialect_text}")
print(f"Normalized (Central Thai) Text: {normalized_text.strip()}")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Local Dialect Text: สวัสดีเจ้า คุณ นาฏฐา ริกา เปิ้น จื้อ เต้ชินี หรือ ได้ ซึ่ง ทราบ ว่า เต้ย ก็ได้ นะเจ้า เปน ตี้ ปรึกษา จาก ธนาคาร ไทย พาณิชย์ เจ้า วัน นี้ เต้ย จะ เข้า ม็อบ เพื่อ ฮึกฮู้ เกี่ยว กับ การ ลอง แผน ทาง การเงิน ตี้ มอง สม กับ เป้าหมาย ของ คุณ นาฏฐา ริกา เจ้า ก็ จะ มี เวลา หื้อ เต้ย สัก ก๋าว ได้ ก่ เจ้า ซัก สาม นาที ก็ได้ เจ้า
Normalized (Central Thai) Text: 


In [15]:
# --- Example with a different dialect (e.g., Northern Thai - Kam Mueang) ---
local_dialect_text_kam_mueang = "สวัสดีเจ้า คุณ นาฏฐา ริกา เปิ้น จื้อ เต้ชินี หรือ ได้ ซึ่ง ทราบ ว่า เต้ย ก็ได้ นะเจ้า เปน ตี้ ปรึกษา จาก ธนาคาร ไทย พาณิชย์ เจ้า วัน นี้ เต้ย จะ เข้า ม็อบ เพื่อ ฮึกฮู้ เกี่ยว กับ การ ลอง แผน ทาง การเงิน ตี้ มอง สม กับ เป้าหมาย ของ คุณ นาฏฐา ริกา เจ้า ก็ จะ มี เวลา หื้อ เต้ย สัก ก๋าว ได้ ก่ เจ้า ซัก สาม นาที ก็ได้ เจ้า" # "Where have you been?" (Northern Thai)
user_prompt_content_kam_mueang = f"Translate the following local dialect text to Central Thai: \"{local_dialect_text_kam_mueang}\""

messages_kam_mueang = [
    {"role": "system", "content": system_prompt_content},
    {"role": "user", "content": user_prompt_content_kam_mueang},
]

input_ids_kam_mueang = tokenizer.apply_chat_template(
    messages_kam_mueang,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

outputs_kam_mueang = model.generate(
    input_ids_kam_mueang,
    max_new_tokens=100,
    eos_token_id=terminators,
    do_sample=True,
    temperature=0.3,
    top_p=0.9,
    pad_token_id=tokenizer.eos_token_id
)
response_ids_kam_mueang = outputs_kam_mueang[input_ids_kam_mueang.shape[-1]:]
normalized_text_kam_mueang = tokenizer.decode(response_ids_kam_mueang, skip_special_tokens=True)

print(f"\nLocal Dialect Text (Kam Mueang): {local_dialect_text_kam_mueang}")
print(f"Normalized (Central Thai) Text: {normalized_text_kam_mueang.strip()}")


Local Dialect Text (Kam Mueang): สวัสดีเจ้า คุณ นาฏฐา ริกา เปิ้น จื้อ เต้ชินี หรือ ได้ ซึ่ง ทราบ ว่า เต้ย ก็ได้ นะเจ้า เปน ตี้ ปรึกษา จาก ธนาคาร ไทย พาณิชย์ เจ้า วัน นี้ เต้ย จะ เข้า ม็อบ เพื่อ ฮึกฮู้ เกี่ยว กับ การ ลอง แผน ทาง การเงิน ตี้ มอง สม กับ เป้าหมาย ของ คุณ นาฏฐา ริกา เจ้า ก็ จะ มี เวลา หื้อ เต้ย สัก ก๋าว ได้ ก่ เจ้า ซัก สาม นาที ก็ได้ เจ้า
Normalized (Central Thai) Text: 
